# FlashVSR-Pro on Google Colab

Select **Runtime → Change runtime type → A100 or L4 GPU**, then run these cells in order. T4/P100 are unsupported. This setup uses Python 3.13–3.14 and CUDA Toolkit 12.8 or newer (12.x). If Colab supplies Python 3.12, select a Python 3.13+ runtime before running these cells.

The installation and inference use a dedicated Python environment in the checkout, separate from the notebook kernel. Start with a short video. Save your output before the runtime expires.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

project = Path('/content/FlashVSR-Pro')
repo_url = 'https://github.com/delvilo/FlashVSR-Pro.git'
branch = 'main'  # Set a feature branch here when testing an unmerged change.

if not (3, 13) <= sys.version_info[:2] < (3, 15):
    raise RuntimeError('Use a runtime with Python 3.13–3.14 for the pinned dependencies.')
subprocess.run([sys.executable, '-c', """
import torch
if not torch.cuda.is_available():
    raise SystemExit('Select a GPU runtime before continuing.')
major, minor = torch.cuda.get_device_capability()
if major not in (8, 9):
    raise SystemExit('Select A100 or L4. This baseline does not support T4/P100.')
print(torch.cuda.get_device_name(), 'compute capability', major, minor)
"""], check=True)
subprocess.run(['nvcc', '--version'], check=True)


## 1. Install the project

CUDA must already be installed in the runtime. If `nvcc` points to an incompatible version, select an installed CUDA 12.8+ toolkit (12.x) using `install_env['CUDA_HOME']` below. CUDA 13.x cannot build against the pinned cu128 wheel. The installer validates the selected toolkit before installing packages. Compilation may take several minutes.


In [ ]:
subprocess.run(['apt-get', 'update'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'build-essential', 'git', 'git-lfs', 'ffmpeg', 'python3-dev', 'python3-venv'], check=True)
if not project.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', branch, repo_url, str(project)], check=True)

venv = project / '.venv'
if not (venv / 'bin/python').exists():
    subprocess.run([sys.executable, '-m', 'venv', str(venv)], check=True)
project_python = str(venv / 'bin/python')
install_env = os.environ.copy()
install_env['FLASHVSR_PYTHON'] = project_python
# install_env['CUDA_HOME'] = '/usr/local/cuda-12.8'
subprocess.run(['bash', 'scripts/install.sh'], cwd=project, env=install_env, check=True)


## 2. Download model weights

The model command downloads the pinned version, verifies size and SHA-256, and copies the packaged prompt into the same cache. Default: `~/.cache/flashvsr/v1.1`. Set `os.environ["FLASHVSR_CACHE_DIR"]` to a mounted Drive directory before this cell to reuse weights across sessions.


In [ ]:
subprocess.run([project_python, '-m', 'flashvsr', 'models', 'download', '--mode', 'all'], cwd=project, check=True)
subprocess.run([project_python, '-m', 'flashvsr', 'models', 'check', '--mode', 'all'], cwd=project, check=True)


## 3. Choose a video

Use the Files panel to upload a video to `/content`, or mount Google Drive and set its full path. Configure the input and output paths below. The default sample is downloaded and verified on demand. Selecting your own video skips the sample download.


In [ ]:
input_video = project / 'inputs/example0.mp4'  # For example: Path('/content/input.mp4')
output_video = Path('/content/enhanced.mp4')
mode = 'tiny'  # full, tiny, tiny-long
scale = '4.0'

if input_video == project / 'inputs/example0.mp4':
    subprocess.run([
        project_python, str(project / 'scripts/download_samples.py'), 'example0.mp4'
    ], cwd=project, check=True)
if not input_video.is_file():
    raise FileNotFoundError(f'Upload a video and set input_video: {input_video}')
output_video.parent.mkdir(parents=True, exist_ok=True)


## 4. Run super-resolution

The default segmented workflow holds at most 129 input frames at a time and reuses one loaded model. Lower `segment_frames` for less VRAM. For full mode, the cell also enables tiled VAE decoding. Set `resume_job = True` after interruption and keep the same settings. Completed segments survive failures; they are removed after success.

For resume across Colab runtime resets, place the input, output and `job_dir` on mounted Drive and recreate the same Python/package environment. Local `/content` files disappear on reset. A changed input or inference setting needs a new job directory.

Each output has a JSON report (`OUTPUT.json`) with parameters, model version, timings, FPS and peak VRAM. Copy it together with the video before the runtime ends. Add `--log-file /content/run.jsonl` for structured logs.


In [ ]:
use_segments = True
segment_frames = 129
resume_job = False  # Set True when continuing the SAME job.
job_dir = output_video.parent / 'flashvsr-jobs' / input_video.stem
if use_segments:
    output_video = output_video.parent / f'FlashVSR_{input_video.stem}_Final.mp4'
    command = [project_python, '-m', 'flashvsr', 'long', '-i', str(input_video),
               '-o', str(output_video.parent), '--work-dir', str(job_dir),
               '--segment-frames', str(segment_frames)]
    if resume_job:
        command.append('--resume')
else:
    command = [project_python, '-m', 'flashvsr', 'infer', '-i', str(input_video), '-o', str(output_video)]
command += ['--mode', mode, '--scale', scale, '--tile-dit', '--keep-audio']
if mode == 'full':
    command.append('--tile-vae')
subprocess.run(command, cwd=project, check=True)
if not output_video.is_file():
    raise FileNotFoundError(f'Expected output was not created: {output_video}')
print(f'Output: {output_video}')


## 5. Download the output

For large files, set `output_video` to a path on mounted Google Drive instead. Colab's local files disappear when the runtime is reset.


In [ ]:
from google.colab import files
files.download(str(output_video))


## 6. Optional GPU acceptance

This runs real inference in all three modes on a generated 17-frame video with audio, then tests two bounded segments and completed-job resume. It checks 2× resolution, frame count, audio tracks, GPU memory measurements and model reuse. It requires all model weights and may take several minutes. Reports and per-mode logs are saved under `results/gpu-validation/`; copy them to Drive before the runtime ends.


In [ ]:
run_gpu_acceptance = False  # Set True to validate all three modes.
if run_gpu_acceptance:
    subprocess.run([project_python, str(project / 'scripts/validate_gpu.py'), '--include-long'], cwd=project, check=True)
